# Figure S1H — Estimated AWS Bedrock Cost by Model (Full Cohort)

Bar chart showing estimated LLM inference cost (USD) for the full patient cohort across 4 foundation models. All values are pre-computed constants — no input data files required.

Outputs are written to `figure 1/results/supp/`.


In [ ]:
import os
from pathlib import Path
import numpy as np
import pandas as pd
import matplotlib
import matplotlib.pyplot as plt
from sklearn.metrics import f1_score

%matplotlib inline

import matplotlib.font_manager as fm
_arial_path = fm.findfont('Arial', fallback_to_default=False)
if 'Arial' not in _arial_path:
    raise RuntimeError(
        f"Arial not found -- matplotlib resolved to '{_arial_path}' instead. "
        "Install Arial or update font.sans-serif before rendering this figure."
    )
print(f"Arial resolved to: {_arial_path}")

In [ ]:
ROOT = Path("..").resolve()
RESULTS = ROOT / "results"
(RESULTS / "supp").mkdir(parents=True, exist_ok=True)

OUT_PATH = RESULTS / "supp" / "Model_Comparison_Cost_S1H.pdf"

In [ ]:
# ---------------------------------------------------------------------------
# CONSTANTS — estimated cost on 70k patients (estimated/extrapolated from Model Costs on smaller subsets)
# ---------------------------------------------------------------------------
COHORT_COSTS_USD = {
    "Llama 4 Maverick":      2904.77,
    "Llama 4 Scout":         2891.19,
    "Claude Sonnet 4.5":     59584.24,
    "GPT-OSS-120B":          2954.01,
}

MODEL_ORDER = ["Llama 4 Maverick", "Llama 4 Scout", "Claude Sonnet 4.5", "GPT-OSS-120B"]
MODEL_COLORS = {
    "Llama 4 Maverick": "#4878CF",
    "Llama 4 Scout":    "#D65F5F",
    "Claude Sonnet 4.5": "#6ACC64",
    "GPT-OSS-120B":     "#B47CC7",
}


In [ ]:
# ---------------------------------------------------------------------------
# Figure
# ---------------------------------------------------------------------------
# ---------------------------------------------------------------------------
# Figure
# ---------------------------------------------------------------------------
plt.rcParams.update({
    "font.family": "Arial",
    "font.size": 6,
    "axes.labelsize": 7,
    "xtick.labelsize": 6,
    "ytick.labelsize": 6,
    "pdf.fonttype": 42,
    "ps.fonttype": 42,
})

labels = MODEL_ORDER
values = [COHORT_COSTS_USD[m] for m in labels]
colors = [MODEL_COLORS[m] for m in labels]
x = range(len(labels))

fig, ax = plt.subplots(figsize=(3.9, 2.5))  

bars = ax.bar(
    x, values,
    color=colors, edgecolor="white", linewidth=0.4,
    width=0.68, alpha=0.9,
)

ax.set_ylabel("Estimated cohort cost\n(USD)", labelpad=6)
ax.set_xticks(list(x))
ax.set_xticklabels(labels, rotation=20, ha="right")  # rotated -- narrower canvas than before
ax.set_ylim(0, max(values) * 1.12)

for spine in ("top", "right"):
    ax.spines[spine].set_visible(False)
for spine in ("bottom", "left"):
    ax.spines[spine].set_linewidth(0.6)
ax.tick_params(width=0.6, length=3)

ymax = max(values)
for bar, v in zip(bars, values):
    h = bar.get_height()
    ax.text(
        bar.get_x() + bar.get_width() / 2.0,
        h + ymax * 0.015,
        f"${v:,.2f}",
        ha="center", va="bottom", fontsize=5,
    )

# Same top/bottom fractions as G -- this is what makes the x-axis line and axes
# box height match when the two panels sit side by side. Left/right differ since
# H's y-axis labels (dollar amounts, 2-line title) need more horizontal room than G's.
fig.subplots_adjust(left=0.22, right=0.97, top=0.94, bottom=0.30)
fig.savefig(OUT_PATH, format="pdf", dpi=450)
print(f"Saved: {OUT_PATH.name}")
plt.show()

In [ ]:
CSV_OUT = OUT_PATH.with_name(OUT_PATH.stem + "_results.csv")

export_df = pd.DataFrame({
    "model": MODEL_ORDER,
    "estimated_cohort_cost_usd": [COHORT_COSTS_USD[m] for m in MODEL_ORDER],
})

cheapest = export_df["estimated_cohort_cost_usd"].min()
export_df["cost_ratio_vs_cheapest"] = export_df["estimated_cohort_cost_usd"] / cheapest
export_df["plot_order"] = range(1, len(MODEL_ORDER) + 1)

export_df.to_csv(CSV_OUT, index=False)
print(f"Saved: {CSV_OUT.name}")
print(f"  {len(export_df)} models, range ${cheapest:,.2f}–"
      f"${export_df['estimated_cohort_cost_usd'].max():,.2f}")